<a href="https://colab.research.google.com/github/tasyriqomar/ColabMD-Edu_Protein-NA/blob/main/Google_Colab_Worker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 🚀 Initialize Setup and Start Service { display-mode: "form" }
# @markdown Enter your Ngrok Auth Token below, then click the play button to run.
ngrok_auth_token = "" # @param {type:"string"}

# ==============================================================================
# 1. CLEAN WORKSPACE AND PREPARE DIRECTORIES
# ==============================================================================
!rm -rf /app/tools /app/colabmd_service_Protein-Nucleic-Acid
!mkdir -p /app/tools /app/shared_data
%cd /app/tools

# ==============================================================================
# 2. CLONE PROTEIN-NA REPOSITORY & SETUP TOOLS
# ==============================================================================
!git clone https://github.com/tasyriqomar/ColabMD-Edu_Protein-NA.git temp_repo
!mv temp_repo/Aptamer-Processing /app/tools/
!mv temp_repo/MD/CGMD /app/tools/
!mv temp_repo/Backmapping /app/tools/
!mv temp_repo/Analysis /app/tools/
!rm -rf temp_repo

# Set executable permissions on scripts
!chmod -R +x /app/tools/Backmapping/
!chmod -R +x /app/tools/Aptamer-Processing/
!chmod -R +x /app/tools/Analysis/

# ==============================================================================
# 3. INSTALL SYSTEM DEPENDENCIES (WITHOUT APT PYTHON2)
# ==============================================================================
!apt-get update && apt-get install -y openbabel redis-server dssp gromacs > /dev/null

# ==============================================================================
# 4. INSTALL MICROMAMBA, PYTHON 2.7, AMBERTOOLS & GMX_MMPBSA
# ==============================================================================
!wget -q https://micro.mamba.pm/api/micromamba/linux-64/latest -O micromamba.tar.bz2
!tar -xjf micromamba.tar.bz2 bin/micromamba
!mv bin/micromamba /usr/local/bin/micromamba
!rm -rf micromamba.tar.bz2 bin

# 4a. Install Python 2.7 via micromamba for legacy CGMD scripts
!micromamba create -p /root/py2 -c conda-forge python=2.7 -y > /dev/null
!ln -sf /root/py2/bin/python2 /usr/local/bin/python2
!ln -sf /root/py2/bin/python2.7 /usr/local/bin/python2.7

# 4b. Install AmberTools & gmx_MMPBSA
!micromamba create -p /root/micromamba -c conda-forge ambertools gmx_mmpbsa -y > /dev/null
!ln -sf /root/micromamba/bin/* /usr/local/bin/ 2>/dev/null || true

# ==============================================================================
# 5. INSTALL PYTHON 3 PACKAGES (PLIP, HADDOCK3, Analysis tools)
# ==============================================================================
!pip install -q fastapi uvicorn python-multipart celery redis pyngrok networkx pandas vermouth ttclust plip openpyxl seaborn biopython pdbfixer openmm haddock3 pdb-tools requests

# ==============================================================================
# 6. CLONE AND SETUP BACKEND SERVICE
# ==============================================================================
%cd /app
!rm -rf colabmd_service_Protein-Nucleic-Acid
# ---> REPLACE WITH YOUR GITHUB USERNAME / ACCESS TOKEN IF PRIVATE <---
!git clone https://github.com/tasyriqomar/colabmd_service_Protein-Nucleic-Acid.git
!pip install -q -r colabmd_service_Protein-Nucleic-Acid/requirements.txt

# ==============================================================================
# 7. START BACKGROUND SERVICES & NGROK
# ==============================================================================
import os, time

# Set headless Matplotlib backend and Redis URL globally
os.environ["MPLBACKEND"] = "Agg"
os.environ["REDIS_URL"] = "redis://127.0.0.1:6379/0"

# Kill old processes to prevent port conflict
get_ipython().system_raw('pkill -f uvicorn')
get_ipython().system_raw('pkill -f celery')
get_ipython().system_raw('service redis-server start')

# Start Celery and Uvicorn
get_ipython().system_raw('cd /app/colabmd_service_Protein-Nucleic-Acid && celery -A tasks.worker.celery_app worker --concurrency=2 --loglevel=info > /app/celery.log 2>&1 &')
get_ipython().system_raw('cd /app/colabmd_service_Protein-Nucleic-Acid && uvicorn main:app --host 0.0.0.0 --port 8000 > /app/uvicorn.log 2>&1 &')

time.sleep(5)

# Connect Ngrok using standard port syntax
from pyngrok import ngrok
ngrok.kill()

# Use the token provided in the Colab Form UI
if not ngrok_auth_token:
    print("❌ ERROR: Please paste your Ngrok token in the form above before running.")
else:
    ngrok.set_auth_token(ngrok_auth_token)
    public_url = ngrok.connect(8000).public_url

    print("====================================================")
    print(f"✅ COPY THIS URL: {public_url}/api/v1")
    print("====================================================")